### performance Machine

In [1]:
import os
import re
import pandas as pd

input_folder = r"C:\Users\AsreRayaneh\OneDrive\Desktop\propozal\rahdari\data\perform"
output_folder = r"C:\Users\AsreRayaneh\OneDrive\Desktop\propozal\rahdari\data\performance_files_with_year_month"

os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(input_folder):

    if not file.endswith(".xlsx"):
        continue

    match = re.search(r"performance_(\d{4})_(\d{2})", file)
    if not match:
        print(f"Skipped (name not matched): {file}")
        continue

    year, month = match.groups()
    year_month_jalali = f"{year}-{month}"

    # ✅ Correct reading
    df = pd.read_excel(
        os.path.join(input_folder, file),
        skiprows=4,
        header=0
    )

    # Add column
    df["year_month_jalali"] = year_month_jalali

    # Save
    output_path = os.path.join(output_folder, file)
    df.to_excel(output_path, index=False)

    print(f"Processed correctly: {file}")


Processed correctly: performance_1400_03.xlsx
Processed correctly: performance_1400_07.xlsx
Processed correctly: performance_1400_08.xlsx
Processed correctly: performance_1400_09.xlsx
Processed correctly: performance_1400_10.xlsx
Processed correctly: performance_1400_11.xlsx
Processed correctly: performance_1400_12.xlsx
Processed correctly: performance_1401_01.xlsx
Processed correctly: performance_1401_02.xlsx
Processed correctly: performance_1401_03.xlsx
Processed correctly: performance_1401_04.xlsx
Processed correctly: performance_1401_05.xlsx
Processed correctly: performance_1401_06.xlsx
Processed correctly: performance_1401_07.xlsx
Processed correctly: performance_1401_08.xlsx
Processed correctly: performance_1401_09.xlsx
Processed correctly: performance_1401_10.xlsx
Processed correctly: performance_1401_11.xlsx
Processed correctly: performance_1401_12.xlsx
Processed correctly: performance_1402_01.xlsx
Processed correctly: performance_1402_02.xlsx
Processed correctly: performance_1

In [3]:
import os
import pandas as pd

# --- Configuration ---
input_folder = r"C:\Users\AsreRayaneh\OneDrive\Desktop\propozal\rahdari\data\performance_files_with_year_month"
output_folder = r"C:\Users\AsreRayaneh\OneDrive\Desktop\propozal\rahdari\data\performance_files_summed_with_base"
os.makedirs(output_folder, exist_ok=True)

output_filename = "FINAL_ALL_FILES_COMBINED.xlsx"  # <--- Name of the single output file
base_filename = "performance_1400_03.xlsx"
cols_to_sum = ["Mileage (KM)", "Travel time", "On time (minutes)"]

def prep_numeric_cols(df, columns):
    """Ensures specific columns are numeric and NaNs are replaced with 0."""
    for col in columns:
        if col not in df.columns:
            df[col] = 0
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    return df

def clean_ids(df):
    """Standardizes Machine code and GPS Code to strings for matching."""
    for col in ["Machine code", "GPS Code"]:
        if col not in df.columns:
            df[col] = None
        else:
            # Convert to string, strip whitespace, replace 'nan' text with None
            df[col] = df[col].astype(str).str.strip()
            df[col] = df[col].replace(["nan", "NaN", ""], None)
    return df

# ==========================================
# 1️⃣ Load and Prepare the BASE File ONCE
# ==========================================
print(f"📖 Loading Base File (Source of Truth): {base_filename}")
base_path = os.path.join(input_folder, base_filename)
base_df = pd.read_excel(base_path)

# (Optional) Clean rows 1-4 if needed
if base_df.shape[0] > 4:
     base_df = base_df.iloc[4:].reset_index(drop=True)

base_df = clean_ids(base_df)
base_df = prep_numeric_cols(base_df, cols_to_sum)

# --- CREATE LOOKUP TABLES ---
print("⚙️  Creating Base Lookup Indexes...")
base_by_machine = base_df.groupby("Machine code")[cols_to_sum].sum()
base_by_gps = base_df.groupby("GPS Code")[cols_to_sum].sum()

print("✅ Base file ready.")

# ==========================================
# 2️⃣ Loop through OTHER files (Targets)
# ==========================================

all_processed_data = []  # <--- List to store all dataframes

for file in os.listdir(input_folder):
    if (file == base_filename or 
        not file.endswith(".xlsx") or 
        file.startswith("~$")):
        continue

    print(f"⚡ Processing Target File: {file}")

    try:
        # Load the Target File
        file_path = os.path.join(input_folder, file)
        target_df = pd.read_excel(file_path)

        # (Optional) Clean rows 1-4 if needed
        if target_df.shape[0] > 4:
             target_df = target_df.iloc[4:].reset_index(drop=True)

        # Clean IDs and Numbers
        target_df = clean_ids(target_df)
        target_df = prep_numeric_cols(target_df, cols_to_sum)

        # ==========================================
        # 3️⃣ The Priority Lookup Logic
        # ==========================================
        
        # Step A: Find matches using MACHINE CODE
        match_machine = target_df.join(base_by_machine, on="Machine code", rsuffix="_base_m")

        # Step B: Find matches using GPS CODE
        match_gps = target_df.join(base_by_gps, on="GPS Code", rsuffix="_base_g")

        # Step C: Combine and Sum
        for col in cols_to_sum:
            col_base_m = f"{col}_base_m"
            col_base_g = f"{col}_base_g"
            
            # Extract values safely
            val_m = match_machine[col_base_m] if col_base_m in match_machine.columns else pd.Series(None, index=target_df.index)
            val_g = match_gps[col_base_g] if col_base_g in match_gps.columns else pd.Series(None, index=target_df.index)

            # PRIORITY LOGIC: Machine > GPS > 0
            final_base_value = val_m.combine_first(val_g).fillna(0)

            # Add this base value to the Target's original value
            target_df[col] = target_df[col] + final_base_value

        # ==========================================
        # 4️⃣ Store for Final Combination
        # ==========================================
        
        # Add a column so you know which file this row came from
        target_df["Source_File"] = file
        
        # Append to list instead of saving immediately
        all_processed_data.append(target_df)

    except Exception as e:
        print(f"   ❌ Error processing {file}: {e}")

# ==========================================
# 5️⃣ Concatenate and Save ONE File
# ==========================================
if all_processed_data:
    print("∑ Combining all files into one...")
    
    # Stack all dataframes on top of each other
    final_master_df = pd.concat(all_processed_data, ignore_index=True)
    
    final_output_path = os.path.join(output_folder, output_filename)
    final_master_df.to_excel(final_output_path, index=False)
    
    print(f"🎉 SUCCESS! All data saved to: {final_output_path}")
else:
    print("⚠️ No data was processed.")

📖 Loading Base File (Source of Truth): performance_1400_03.xlsx
⚙️  Creating Base Lookup Indexes...
✅ Base file ready.
⚡ Processing Target File: performance_1400_07.xlsx
⚡ Processing Target File: performance_1400_08.xlsx
⚡ Processing Target File: performance_1400_09.xlsx
⚡ Processing Target File: performance_1400_10.xlsx
⚡ Processing Target File: performance_1400_11.xlsx
⚡ Processing Target File: performance_1400_12.xlsx
⚡ Processing Target File: performance_1401_01.xlsx
⚡ Processing Target File: performance_1401_02.xlsx
⚡ Processing Target File: performance_1401_03.xlsx
⚡ Processing Target File: performance_1401_04.xlsx
⚡ Processing Target File: performance_1401_05.xlsx
⚡ Processing Target File: performance_1401_06.xlsx
⚡ Processing Target File: performance_1401_07.xlsx
⚡ Processing Target File: performance_1401_08.xlsx
⚡ Processing Target File: performance_1401_09.xlsx
⚡ Processing Target File: performance_1401_10.xlsx
⚡ Processing Target File: performance_1401_11.xlsx
⚡ Processing T